# HADO Player Detection — YOLOv8 Fine-tuning

**목표**: AR 이펙트(쉴드·볼·파티클)가 포함된 경기 영상에서 선수를 강건하게 감지하도록 YOLOv8n을 파인튜닝

**런타임**: GPU 필수 (런타임 → 런타임 유형 변경 → T4 GPU)

---
## 순서
1. 환경 설정
2. Roboflow에서 데이터셋 다운로드
3. YOLOv8n 파인튜닝
4. 결과 확인
5. 모델 저장 (Google Drive)

## 0. Google Drive 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/hado_model'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'저장 경로: {SAVE_DIR}')

## 1. 패키지 설치

In [ ]:
!pip install ultralytics roboflow -q

import torch
print(f'CUDA: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음"}')

## 2. Roboflow 데이터셋 다운로드

**사전 준비**:
1. [roboflow.com](https://roboflow.com) 에서 프로젝트 생성
2. 407장 업로드 → 라벨링 완료
3. Export → YOLOv8 포맷
4. API Key + 프로젝트 정보 아래에 입력

In [ ]:
from roboflow import Roboflow

# ── 여기에 본인 정보 입력 ──────────────────────────────────
ROBOFLOW_API_KEY  = 'YOUR_API_KEY'   # roboflow.com → Settings → API Keys
WORKSPACE_NAME    = 'YOUR_WORKSPACE' # URL에서 확인 (예: piaojinu)
PROJECT_NAME      = 'hado-player'    # 프로젝트 이름
VERSION           = 3                # 버전 번호
# ──────────────────────────────────────────────────────────

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE_NAME).project(PROJECT_NAME)
dataset = project.version(VERSION).download('yolov8', location='/content/hado_dataset')

print(f'데이터셋 경로: {dataset.location}')

In [ ]:
# 데이터셋 구성 확인
import os
base = dataset.location
for split in ['train', 'valid', 'test']:
    img_dir = os.path.join(base, split, 'images')
    if os.path.exists(img_dir):
        n = len(os.listdir(img_dir))
        print(f'  {split}: {n}장')

## 3. YOLOv8n 파인튜닝

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # 사전학습 가중치 로드

results = model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=60,
    imgsz=640,
    batch=16,           # T4 기준 적정값. OOM 나면 8로 줄이기
    patience=15,        # 15에폭 개선 없으면 조기 종료
    device=0,
    # ── augmentation (AR 이펙트 다양성 보완) ──
    hsv_h=0.02,         # 색조 변화 (쉴드 색상 변형 시뮬레이션)
    hsv_s=0.8,
    hsv_v=0.5,
    mosaic=1.0,         # 4장 합성 → 다양한 구도
    mixup=0.1,
    copy_paste=0.2,     # 선수를 다른 배경에 복사
    flipud=0.0,         # 상하 반전 없음 (코트 위아래 고정)
    fliplr=0.5,
    project='runs/hado',
    name='finetune_v3',
    save=True,
)

## 4. 학습 결과 확인

In [ ]:
from IPython.display import Image, display

# 학습 곡선
display(Image('runs/hado/finetune_v3/results.png'))

In [ ]:
# validation 혼동 행렬
display(Image('runs/hado/finetune_v3/confusion_matrix.png'))

In [ ]:
# best.pt로 validation 재평가
best_model = YOLO('runs/hado/finetune_v3/weights/best.pt')
metrics = best_model.val(data=f'{dataset.location}/data.yaml', imgsz=640)
print(f'mAP50     : {metrics.box.map50:.3f}')
print(f'mAP50-95  : {metrics.box.map:.3f}')
print(f'Precision : {metrics.box.mp:.3f}')
print(f'Recall    : {metrics.box.mr:.3f}')

## 5. 영상에서 테스트 추론

In [ ]:
# 원본 yolov8n vs 파인튜닝 비교
import cv2
import numpy as np
from google.colab.patches import cv2_imshow

# 테스트 이미지 (validation 셋에서 heavy 1장 선택)
import glob, random
heavy_imgs = glob.glob(f'{dataset.location}/valid/images/*heavy*')
if not heavy_imgs:
    heavy_imgs = glob.glob(f'{dataset.location}/valid/images/*.jpg')
test_img = random.choice(heavy_imgs)
print(f'테스트 이미지: {test_img}')

original = YOLO('yolov8n.pt')
finetuned = YOLO('runs/hado/finetune_v3/weights/best.pt')

frame = cv2.imread(test_img)

r_orig = original(frame, conf=0.25, verbose=False)[0]
r_fine = finetuned(frame, conf=0.25, verbose=False)[0]

img_orig = r_orig.plot()
img_fine = r_fine.plot()

# 좌우 비교
h = max(img_orig.shape[0], img_fine.shape[0])
def pad(img, h):
    dh = h - img.shape[0]
    return np.pad(img, ((0,dh),(0,0),(0,0)))

cv2.putText(img_orig, 'Original yolov8n', (10,30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)
cv2.putText(img_fine, 'HADO Fine-tuned v3', (10,30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)
combined = np.hstack([pad(img_orig, h), pad(img_fine, h)])
combined_small = cv2.resize(combined, (1280, int(1280 * combined.shape[0]/combined.shape[1])))
cv2_imshow(combined_small)
print(f'원본: {len(r_orig.boxes)}명  |  파인튜닝: {len(r_fine.boxes)}명')

## 6. 모델 저장

In [ ]:
import shutil

# best.pt → Google Drive 저장
src = 'runs/hado/finetune_v3/weights/best.pt'
dst = f'{SAVE_DIR}/hado_yolov8n_v3.pt'
shutil.copy(src, dst)
print(f'저장 완료: {dst}')

# 학습 결과 그래프도 저장
shutil.copy('runs/hado/finetune_v3/results.png', f'{SAVE_DIR}/results_v3.png')
print('학습 곡선 저장 완료')

## 7. 프로젝트에 모델 적용

Google Drive에서 `hado_yolov8n_v1.pt` 다운로드 후:

```bash
# demo_pose.py 실행 시 모델 교체
python -m src.demo_pose \
  --video 'data/경기영상.mp4' \
  --model hado_yolov8n_v1.pt \
  --imgsz 640 --conf 0.25 --headless \
  --out data/result.mp4
```

> **주의**: 파인튜닝 모델은 `yolov8n.pt` (bbox only). 키포인트는 없으므로 스켈레톤 오버레이는 동작하지 않음.  
> 위치 추적 + 버드아이뷰 + 전술 조언은 정상 동작.